# Experiment 1b — Authority direction from conflict pairs

Uses the 4-trace contrastive design on evolved conflict pairs to find the authority
direction from behavioral differences (system-following vs user-following), not structural position.

Requires a GPU (A100 recommended). Set MODEL_KEY in the config cell, then Run All.

In [ ]:
# Install
import os, sys, importlib, site
REPO_DIR = "/content/Mech_spoof"

if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/ChuloIva/Mech_spoof.git {REPO_DIR}

!pip install -q -e {REPO_DIR}

os.environ["MECH_SPOOF_ROOT"] = REPO_DIR

src_path = f"{REPO_DIR}/src"
if src_path not in sys.path:
    sys.path.insert(0, src_path)
importlib.invalidate_caches()
site.main()

import mech_spoof
print("mech_spoof loaded from:", mech_spoof.__file__)

In [ ]:
# Auth + Drive
import os, getpass
from google.colab import drive, userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
drive.mount("/content/drive")

In [ ]:
# Config (EDIT ME)
from pathlib import Path
from mech_spoof.configs import MODEL_CONFIGS

MODEL_KEY = "qwen"            # qwen | llama3 | mistral | gemma | phi3 | gemma_small
DRIVE_ROOT = Path("/content/drive/MyDrive/mech_spoof_results")

slug = MODEL_CONFIGS[MODEL_KEY].slug
OUT_DIR = DRIVE_ROOT / slug / "exp1b_authority_conflict"
OUT_DIR.mkdir(parents=True, exist_ok=True)

EXP1_DIR = DRIVE_ROOT / slug / "exp1_authority"  # for cross-check (optional)

print(f"Model: {MODEL_KEY} ({MODEL_CONFIGS[MODEL_KEY].hf_id})")
print(f"OUT_DIR = {OUT_DIR}")
print(f"EXP1_DIR = {EXP1_DIR} (exists={EXP1_DIR.exists()})")

In [ ]:
# Run experiment 1b
from mech_spoof.experiments.exp1b_authority_conflict import run_experiment_1b

result = run_experiment_1b(
    MODEL_KEY,
    OUT_DIR,
    exp1_dir=EXP1_DIR if EXP1_DIR.exists() else None,
    batch_size=4,       # bump to 8-16 on A100 80GB
    max_length=2048,
    max_response_chars=3000,
)
print(f"\nBest layer: {result.best_layer}")
print(f"Best accuracy: {result.best_accuracy:.4f}")

## Results

In [ ]:
# Per-layer accuracy plot
import matplotlib.pyplot as plt
import numpy as np

n_layers = result.n_layers
layers = list(range(n_layers))
accs = [result.accuracies[l] for l in layers]
aurocs = [result.aurocs[l] for l in layers]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(layers, accs, marker="o", markersize=3, label="Accuracy")
ax1.axhline(0.5, color="gray", ls="--", lw=0.8, label="Chance")
ax1.axvline(result.best_layer, color="red", ls=":", alpha=0.7, label=f"Best: L{result.best_layer}")
ax1.set_xlabel("Layer")
ax1.set_ylabel("Probe accuracy")
ax1.set_ylim(0.4, 1.05)
ax1.set_title(f"{MODEL_KEY}: exp1b per-layer probe accuracy")
ax1.legend()

ax2.plot(layers, aurocs, marker="s", markersize=3, color="orange", label="AUROC")
ax2.axhline(0.5, color="gray", ls="--", lw=0.8)
ax2.axvline(result.best_layer, color="red", ls=":", alpha=0.7)
ax2.set_xlabel("Layer")
ax2.set_ylabel("AUROC")
ax2.set_ylim(0.4, 1.05)
ax2.set_title(f"{MODEL_KEY}: exp1b per-layer AUROC")
ax2.legend()

plt.tight_layout()
plt.savefig(OUT_DIR / "layer_accuracy.png", dpi=120)
plt.show()

In [ ]:
# Probe vs diff-in-means agreement
cosines = [result.probe_vs_dim_cosine[l] for l in layers]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(layers, cosines, marker=".", color="green")
ax.set_xlabel("Layer")
ax.set_ylabel("Cosine similarity")
ax.set_title(f"{MODEL_KEY}: probe direction vs diff-in-means agreement")
ax.set_ylim(-0.1, 1.1)
plt.tight_layout()
plt.savefig(OUT_DIR / "probe_vs_dim_cosine.png", dpi=120)
plt.show()

In [ ]:
# Per macro-axis breakdown at best layer
breakdown = result.macro_axis_breakdown
print(f"Best layer: {result.best_layer}  |  Overall accuracy: {result.best_accuracy:.4f}\n")
print(f"{'Axis':>15s}  {'N':>5s}  {'Acc':>6s}  {'AUROC':>6s}")
print("-" * 38)
for axis in sorted(breakdown, key=lambda a: -breakdown[a]['accuracy']):
    d = breakdown[axis]
    print(f"{axis:>15s}  {d['n']:5d}  {d['accuracy']:6.3f}  {d['auroc']:6.3f}")

In [ ]:
# Cross-check with exp1 structural direction (if available)
if result.exp1_cosine_agreement is not None:
    exp1_cos = [result.exp1_cosine_agreement.get(l, float('nan')) for l in layers]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(layers, exp1_cos, marker='.', color='purple')
    ax.set_xlabel('Layer')
    ax.set_ylabel('Cosine similarity')
    ax.set_title(f'{MODEL_KEY}: exp1b direction vs exp1 structural direction')
    ax.set_ylim(-1.1, 1.1)
    ax.axhline(0, color='gray', ls='--', lw=0.8)
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'exp1b_vs_exp1_cosine.png', dpi=120)
    plt.show()
    cos_at_best = result.exp1_cosine_agreement.get(result.best_layer, float('nan'))
    print(f'Cosine at best layer (L{result.best_layer}): {cos_at_best:.4f}')
else:
    print('No exp1 results found — skipping cross-check.')

In [ ]:
# Summary
print(f"Model:          {result.model_key}")
print(f"Layers:         {result.n_layers}")
print(f"d_model:        {result.d_model}")
print(f"Pairs:          {result.n_pairs}")
print(f"Traces:         {result.n_system_following + result.n_user_following} ({result.n_system_following} sys + {result.n_user_following} usr)")
print(f"Best layer:     {result.best_layer}")
print(f"Best accuracy:  {result.best_accuracy:.4f}")
print(f"\nResults saved to: {OUT_DIR}")